In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer

In [2]:
import sys
sys.path.append("./cluster_anlys")

from kondrak_cluster_morphology import (
    run_kondrak_morphology_for_partition,   
    run_kondrak_morphology_for_random_pca_partition,
    concat_global_summary_rows,
    load_random_pca_summary_df,
)

seed_list = [0, 42, 1000, 9999, 1813382118, 827307999, 1627694678, 1911784257]

In [3]:
seed_list= [1813382118, 827307999, 1627694678, 1911784257]

In [4]:
seed_list = [1813382118,
 827307999,
 1627694678,
 1911784257,
 903170602,
 86939546,
 556019485,
 2073320061,
 1097954097,
 1043521778]

# Mistral-7B

In [5]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "permutation/mistralai/Mistral-7B-v0.1_rand"
space_name = "output_proj"

# out_root = "comp"
# model_name = "mistralai/Mistral-7B-v0.1"
# space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"mistralai/Mistral-7B-v0.1/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096]

[✓] Tokenizer loaded from: mistralai/Mistral-7B-v0.1/tokenizer


In [20]:
summary_df = pd.read_csv(f"{out_root}/{model_name}/{space_name}/summary.csv")
summary_df.head()


,run_id,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,n_clusters_excl_noise,n_noise,n_total,noise_ratio,avg_prob_all,avg_prob_assigned,cluster_csv_path,meta_json_path
0,1,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,5,True,5,5,euclidean,eom,0.0,308,28275,32000,0.883594,0.109885,0.943976,comp/permutation/mistralai/Mistral-7B-v0.1_ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/met...
1,2,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,142,True,5,5,euclidean,eom,0.0,79,29230,32000,0.913438,0.074777,0.863849,comp/permutation/mistralai/Mistral-7B-v0.1_ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/met...
2,3,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,997,True,5,5,euclidean,eom,0.0,647,24663,32000,0.770719,0.212622,0.927340,comp/permutation/mistralai/Mistral-7B-v0.1_ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/met...
3,4,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,2084,True,5,5,euclidean,eom,0.0,883,22037,32000,0.688656,0.293329,0.942140,comp/permutation/mistralai/Mistral-7B-v0.1_ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/met...
4,5,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3031,True,5,5,euclidean,eom,0.0,939,21760,32000,0.680000,0.303131,0.947284,comp/permutation/mistralai/Mistral-7B-v0.1_ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/met...


In [21]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=5 ===

=== Running pca_dim=142 ===

=== Running pca_dim=997 ===

=== Running pca_dim=2084 ===

=== Running pca_dim=3031 ===

=== Running pca_dim=3459 ===

=== Running pca_dim=3938 ===

=== Running pca_dim=4088 ===

=== Running pca_dim=4096 ===


In [22]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,5,True,5,5,euclidean,eom,0.0,0.107258,0.056776,308
1,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,142,True,5,5,euclidean,eom,0.0,0.119065,0.059603,79
2,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,997,True,5,5,euclidean,eom,0.0,0.140577,0.050177,647
3,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,2084,True,5,5,euclidean,eom,0.0,0.145867,0.048397,883
4,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3031,True,5,5,euclidean,eom,0.0,0.151674,0.049508,939
5,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3459,True,5,5,euclidean,eom,0.0,0.155764,0.048622,953
6,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3938,True,5,5,euclidean,eom,0.0,0.155351,0.048279,968
7,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,4088,True,5,5,euclidean,eom,0.0,0.152383,0.048600,971
8,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,4096,True,5,5,euclidean,eom,0.0,0.154526,0.048534,983


In [22]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=1813382118 ===

=== Running pca_seed=1813382118, pca_dim=5 ===


FileNotFoundError: Random PCA summary not found: comp/permutation/mistralai/Mistral-7B-v0.1_rand/output_proj/random/seed_1813382118/summary_random_pca_seed_1813382118.csv

### Permutation

In [6]:
from kondrak_cluster_morphology import run_kondrak_morphology_for_permutation_seeds

permutation_result = run_kondrak_morphology_for_permutation_seeds(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    perm_seed_list=seed_list,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    token_col=token_col,
    cluster_id_col=cluster_id_col,
    ddof=0,
    print_columns=False,
    save_cluster_csv=True,
    tokenizer=tokenizer,
)

permutation_result["mean_summary_df"]


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,5,True,5,5,euclidean,eom,0.0,0.104205,0.052867,308
1,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,142,True,5,5,euclidean,eom,0.0,0.112917,0.058142,79
2,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,997,True,5,5,euclidean,eom,0.0,0.140888,0.050611,647
3,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,2084,True,5,5,euclidean,eom,0.0,0.147486,0.049607,883
4,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3031,True,5,5,euclidean,eom,0.0,0.150275,0.048295,939
5,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3459,True,5,5,euclidean,eom,0.0,0.156213,0.047798,953
6,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,3938,True,5,5,euclidean,eom,0.0,0.155221,0.048088,968
7,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,4088,True,5,5,euclidean,eom,0.0,0.152516,0.048768,971
8,permutation/mistralai/Mistral-7B-v0.1_rand,output_proj,4096,True,5,5,euclidean,eom,0.0,0.153946,0.047586,983


# Mistral-8x7B

In [7]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "permutation/mistralai/Mixtral-8x7B-v0.1_rand"
space_name = "output_proj"

# out_root = "comp"
# model_name = "mistralai/Mixtral-8x7B-v0.1"
# space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"mistralai/Mixtral-8x7B-v0.1/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096]

[✓] Tokenizer loaded from: mistralai/Mixtral-8x7B-v0.1/tokenizer


In [24]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=8 ===

=== Running pca_dim=158 ===

=== Running pca_dim=1111 ===

=== Running pca_dim=2156 ===

=== Running pca_dim=3052 ===

=== Running pca_dim=3457 ===

=== Running pca_dim=3957 ===

=== Running pca_dim=4093 ===

=== Running pca_dim=4096 ===


In [25]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,8,True,5,5,euclidean,eom,0.0,0.155306,0.059515,87
1,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,158,True,5,5,euclidean,eom,0.0,0.103719,0.050075,228
2,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,1111,True,5,5,euclidean,eom,0.0,0.151541,0.047165,978
3,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,2156,True,5,5,euclidean,eom,0.0,0.151584,0.045383,1082
4,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3052,True,5,5,euclidean,eom,0.0,0.152303,0.046345,1098
5,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3457,True,5,5,euclidean,eom,0.0,0.151533,0.047586,1097
6,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3957,True,5,5,euclidean,eom,0.0,0.148723,0.048023,1094
7,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,4093,True,5,5,euclidean,eom,0.0,0.151203,0.048639,1091
8,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,4096,True,5,5,euclidean,eom,0.0,0.150959,0.050174,1091


### random

In [17]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=0 ===

=== Running pca_seed=0, pca_dim=8 ===

=== Running pca_seed=0, pca_dim=158 ===

=== Running pca_seed=0, pca_dim=1111 ===

=== Running pca_seed=0, pca_dim=2156 ===

=== Running pca_seed=0, pca_dim=3052 ===

=== Running pca_seed=0, pca_dim=3457 ===

=== Running pca_seed=0, pca_dim=3957 ===

=== Running pca_seed=0, pca_dim=4093 ===

=== Running pca_seed=0, pca_dim=4096 ===

=== Running pca_seed=42 ===

=== Running pca_seed=42, pca_dim=8 ===

=== Running pca_seed=42, pca_dim=158 ===

=== Running pca_seed=42, pca_dim=1111 ===

=== Running pca_seed=42, pca_dim=2156 ===

=== Running pca_seed=42, pca_dim=3052 ===

=== Running pca_seed=42, pca_dim=3457 ===

=== Running pca_seed=42, pca_dim=3957 ===

=== Running pca_seed=42, pca_dim=4093 ===

=== Running pca_seed=42, pca_dim=4096 ===

=== Running pca_seed=1000 ===

=== Running pca_seed=1000, pca_dim=8 ===

=== Running pca_seed=1000, pca_dim=158 ===

=== Running pca_seed=1000, pca_dim=1111 ===

=== Running pca_seed=10

### Permutation

In [8]:
from kondrak_cluster_morphology import run_kondrak_morphology_for_permutation_seeds

permutation_result = run_kondrak_morphology_for_permutation_seeds(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    perm_seed_list=seed_list,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    token_col=token_col,
    cluster_id_col=cluster_id_col,
    ddof=0,
    print_columns=False,
    save_cluster_csv=True,
    tokenizer=tokenizer,
)

permutation_result["mean_summary_df"]


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,8,True,5,5,euclidean,eom,0.0,0.154477,0.055887,87
1,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,158,True,5,5,euclidean,eom,0.0,0.105777,0.053418,228
2,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,1111,True,5,5,euclidean,eom,0.0,0.151777,0.048003,978
3,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,2156,True,5,5,euclidean,eom,0.0,0.151809,0.046945,1082
4,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3052,True,5,5,euclidean,eom,0.0,0.153449,0.047101,1098
5,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3457,True,5,5,euclidean,eom,0.0,0.152492,0.047223,1097
6,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,3957,True,5,5,euclidean,eom,0.0,0.148313,0.047712,1094
7,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,4093,True,5,5,euclidean,eom,0.0,0.149109,0.048307,1091
8,permutation/mistralai/Mixtral-8x7B-v0.1_rand,output_proj,4096,True,5,5,euclidean,eom,0.0,0.149097,0.047718,1091


# gpt-oss-20B

In [7]:
# =========================================================
# params
# =========================================================
out_root = "comp"
model_name = "permutation/gpt-oss_rand"
space_name = "output_proj"

# out_root = "comp"
# model_name = "gpt-oss"
# space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"gpt-oss/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


l2_norm = True
min_cluster_size = 5
min_samples = 5
metric = "euclidean"
cluster_selection_method = "eom"
cluster_selection_epsilon = 0.0

token_col = "token_str"
cluster_id_col = "cluster_id"

pca_dim_list = [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880]

[✓] Tokenizer loaded from: gpt-oss/tokenizer


In [6]:
# Collect global statistics for each PCA dimension
global_rows = []

for pca_dim in pca_dim_list:
    print(f"\n=== Running pca_dim={pca_dim} ===")

    # Construct output path for cluster-level CSV
    cluster_outpath = (
        f"{out_root}/{model_name}/{space_name}/morph/"
        f"kondrak_clusters_pca_{pca_dim}.csv"
    )

    Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)
    # Run morphology analysis for the current partition
    # Returns:
    #   - cluster_df: per-cluster metrics
    #   - global_row: aggregated statistics for this PCA dimension
    cluster_df, global_row = run_kondrak_morphology_for_partition(
        out_root=out_root,
        model_name=model_name,
        space_name=space_name,
        pca_dim=pca_dim,
        l2_norm=l2_norm,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
        token_col=token_col,
        cluster_id_col=cluster_id_col,
        ddof=0,
        print_columns=False,
        save_cluster_csv=True,               # enable saving cluster-level CSV
        tokenizer=tokenizer,
        cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
    )

    # Store global summary for this PCA dimension
    global_rows.append(global_row)

# Concatenate all global summaries into a single DataFrame
global_summary_df = concat_global_summary_rows(global_rows)

# Sort by PCA dimension for consistent ordering
global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

# Save global summary CSV
global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/kondrak_global_summary.csv",
    index=False
)


=== Running pca_dim=6 ===


KeyboardInterrupt: 

In [ ]:
# =========================================================
# sum
# =========================================================
global_summary_df = concat_global_summary_rows(global_rows)

global_summary_df.sort_values("pca_dim", inplace=True)
global_summary_df.reset_index(drop=True, inplace=True)

global_summary_df

### random

In [19]:
from pathlib import Path

# Collect global statistics for each seed and PCA dimension
all_seed_global_summary = []

for pca_seed in seed_list:
    print(f"\n==============================")
    print(f"=== Running pca_seed={pca_seed} ===")
    print(f"==============================")

    global_rows = []

    for pca_dim in pca_dim_list:
        print(f"\n=== Running pca_seed={pca_seed}, pca_dim={pca_dim} ===")

        # Construct output path for cluster-level CSV
        cluster_outpath = (
            f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/morph/"
            f"kondrak_clusters_pca_{pca_dim}.csv"
        )

        Path(cluster_outpath).parent.mkdir(parents=True, exist_ok=True)

        # Run morphology analysis for the current random-PCA partition
        # Returns:
        #   - cluster_df: per-cluster metrics
        #   - global_row: aggregated statistics for this PCA dimension
        cluster_df, global_row = run_kondrak_morphology_for_random_pca_partition(
            out_root=out_root,
            model_name=model_name,
            space_name=space_name,
            pca_seed=pca_seed,
            pca_dim=pca_dim,
            l2_norm=l2_norm,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric,
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            token_col=token_col,
            cluster_id_col=cluster_id_col,
            ddof=0,
            print_columns=False,
            save_cluster_csv=True,               # enable saving cluster-level CSV
            tokenizer=tokenizer,
            cluster_csv_outpath=cluster_outpath, # output path for cluster CSV
        )

        # Store global summary for this PCA dimension
        global_rows.append(global_row)

    # Concatenate all global summaries for this seed into a single DataFrame
    global_summary_df = concat_global_summary_rows(global_rows)

    # Sort by PCA dimension for consistent ordering
    global_summary_df.sort_values("pca_dim", inplace=True)
    global_summary_df.reset_index(drop=True, inplace=True)

    # Save per-seed global summary CSV
    seed_summary_outpath = (
        f"{out_root}/{model_name}/{space_name}/random/seed_{pca_seed}/"
        f"kondrak_global_summary.csv"
    )
    Path(seed_summary_outpath).parent.mkdir(parents=True, exist_ok=True)
    global_summary_df.to_csv(seed_summary_outpath, index=False)

    all_seed_global_summary.append(global_summary_df)

# Optional: concatenate all seeds into one summary DataFrame
all_seed_global_summary_df = concat_global_summary_rows(all_seed_global_summary)

# Sort for consistent ordering
all_seed_global_summary_df.sort_values(
    ["pca_seed", "pca_dim"], inplace=True
)
all_seed_global_summary_df.reset_index(drop=True, inplace=True)

# Save all-seed summary CSV
all_seed_global_summary_df.to_csv(
    f"{out_root}/{model_name}/{space_name}/random/kondrak_global_summary_all_seeds.csv",
    index=False
)


=== Running pca_seed=0 ===

=== Running pca_seed=0, pca_dim=6 ===

=== Running pca_seed=0, pca_dim=182 ===

=== Running pca_seed=0, pca_dim=466 ===

=== Running pca_seed=0, pca_dim=739 ===

=== Running pca_seed=0, pca_dim=1591 ===

=== Running pca_seed=0, pca_dim=2264 ===

=== Running pca_seed=0, pca_dim=2532 ===

=== Running pca_seed=0, pca_dim=2868 ===

=== Running pca_seed=0, pca_dim=2880 ===

=== Running pca_seed=42 ===

=== Running pca_seed=42, pca_dim=6 ===

=== Running pca_seed=42, pca_dim=182 ===

=== Running pca_seed=42, pca_dim=466 ===

=== Running pca_seed=42, pca_dim=739 ===

=== Running pca_seed=42, pca_dim=1591 ===

=== Running pca_seed=42, pca_dim=2264 ===

=== Running pca_seed=42, pca_dim=2532 ===

=== Running pca_seed=42, pca_dim=2868 ===

=== Running pca_seed=42, pca_dim=2880 ===

=== Running pca_seed=1000 ===

=== Running pca_seed=1000, pca_dim=6 ===

=== Running pca_seed=1000, pca_dim=182 ===

=== Running pca_seed=1000, pca_dim=466 ===

=== Running pca_seed=1000, p

### Permutation

In [8]:
from kondrak_cluster_morphology import run_kondrak_morphology_for_permutation_seeds

permutation_result = run_kondrak_morphology_for_permutation_seeds(
    out_root=out_root,
    model_name=model_name,
    space_name=space_name,
    perm_seed_list=seed_list,
    pca_dim_list=pca_dim_list,
    l2_norm=l2_norm,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric=metric,
    cluster_selection_method=cluster_selection_method,
    cluster_selection_epsilon=cluster_selection_epsilon,
    token_col=token_col,
    cluster_id_col=cluster_id_col,
    ddof=0,
    print_columns=False,
    save_cluster_csv=True,
    tokenizer=tokenizer,
)

permutation_result["mean_summary_df"]


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/gpt-oss_rand,output_proj,6,True,5,5,euclidean,eom,0.0,0.113057,0.047237,1203
1,permutation/gpt-oss_rand,output_proj,182,True,5,5,euclidean,eom,0.0,0.060667,0.030542,740
2,permutation/gpt-oss_rand,output_proj,466,True,5,5,euclidean,eom,0.0,0.088552,0.037852,1941
3,permutation/gpt-oss_rand,output_proj,739,True,5,5,euclidean,eom,0.0,0.102142,0.037081,2783
4,permutation/gpt-oss_rand,output_proj,1591,True,5,5,euclidean,eom,0.0,0.109458,0.034860,3625
5,permutation/gpt-oss_rand,output_proj,2264,True,5,5,euclidean,eom,0.0,0.108611,0.034784,3819
6,permutation/gpt-oss_rand,output_proj,2532,True,5,5,euclidean,eom,0.0,0.109052,0.034582,3856
7,permutation/gpt-oss_rand,output_proj,2868,True,5,5,euclidean,eom,0.0,0.109752,0.034679,3880
8,permutation/gpt-oss_rand,output_proj,2880,True,5,5,euclidean,eom,0.0,0.109599,0.034627,3880


# Length-bucketed permutation morphology


In [10]:
from kondrak_cluster_morphology import run_kondrak_morphology_for_permutation_seeds

length_bucket_seed_list = [
    1813382118,
    827307999,
    1627694678,
    1911784257,
    903170602,
    86939546,
    556019485,
    2073320061,
    1097954097,
    1043521778,
]

length_bucket_common_params = {
    "out_root": "comp",
    "space_name": "output_proj",
    "l2_norm": True,
    "min_cluster_size": 5,
    "min_samples": 5,
    "metric": "euclidean",
    "cluster_selection_method": "eom",
    "cluster_selection_epsilon": 0.0,
    "token_col": "token_str",
    "cluster_id_col": "cluster_id",
    "ddof": 0,
    "print_columns": False,
    "save_cluster_csv": True,
}


# Mistral-7B length-bucketed permutation


In [7]:
model_name = "permutation/mistralai/Mistral-7B-v0.1_length_bucket_rand"
tokenizer_path = "mistralai/Mistral-7B-v0.1/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

mistral_length_bucket_morph_result = run_kondrak_morphology_for_permutation_seeds(
    **length_bucket_common_params,
    model_name=model_name,
    perm_seed_list=length_bucket_seed_list,
    pca_dim_list=[5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096],
    tokenizer=tokenizer,
)

mistral_length_bucket_morph_result["mean_summary_df"]


[✓] Tokenizer loaded from: mistralai/Mistral-7B-v0.1/tokenizer


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,5,True,5,5,euclidean,eom,0.0,0.112214,0.072040,308
1,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,142,True,5,5,euclidean,eom,0.0,0.128083,0.080792,79
2,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,997,True,5,5,euclidean,eom,0.0,0.155561,0.056189,647
3,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,2084,True,5,5,euclidean,eom,0.0,0.156656,0.054875,883
4,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,3031,True,5,5,euclidean,eom,0.0,0.155763,0.054167,939
5,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,3459,True,5,5,euclidean,eom,0.0,0.159409,0.053771,953
6,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,3938,True,5,5,euclidean,eom,0.0,0.158074,0.054962,968
7,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,4088,True,5,5,euclidean,eom,0.0,0.156945,0.054334,971
8,permutation/mistralai/Mistral-7B-v0.1_length_b...,output_proj,4096,True,5,5,euclidean,eom,0.0,0.156513,0.054968,983


# Mixtral-8x7B length-bucketed permutation


In [8]:
model_name = "permutation/mistralai/Mixtral-8x7B-v0.1_length_bucket_rand"
tokenizer_path = "mistralai/Mixtral-8x7B-v0.1/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

mixtral_length_bucket_morph_result = run_kondrak_morphology_for_permutation_seeds(
    **length_bucket_common_params,
    model_name=model_name,
    perm_seed_list=length_bucket_seed_list,
    pca_dim_list=[8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096],
    tokenizer=tokenizer,
)

mixtral_length_bucket_morph_result["mean_summary_df"]


[✓] Tokenizer loaded from: mistralai/Mixtral-8x7B-v0.1/tokenizer


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,8,True,5,5,euclidean,eom,0.0,0.124323,0.087578,87
1,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,158,True,5,5,euclidean,eom,0.0,0.137280,0.063909,228
2,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,1111,True,5,5,euclidean,eom,0.0,0.156776,0.054170,978
3,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,2156,True,5,5,euclidean,eom,0.0,0.155950,0.053116,1082
4,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,3052,True,5,5,euclidean,eom,0.0,0.155934,0.053123,1098
5,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,3457,True,5,5,euclidean,eom,0.0,0.155246,0.053867,1097
6,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,3957,True,5,5,euclidean,eom,0.0,0.155024,0.054170,1094
7,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,4093,True,5,5,euclidean,eom,0.0,0.154605,0.054164,1091
8,permutation/mistralai/Mixtral-8x7B-v0.1_length...,output_proj,4096,True,5,5,euclidean,eom,0.0,0.154542,0.054334,1091


# gpt-oss length-bucketed permutation


In [11]:
model_name = "permutation/gpt-oss_length_bucket_rand"
tokenizer_path = "gpt-oss/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

gpt_oss_length_bucket_morph_result = run_kondrak_morphology_for_permutation_seeds(
    **length_bucket_common_params,
    model_name=model_name,
    perm_seed_list=length_bucket_seed_list,
    pca_dim_list=[6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880],
    tokenizer=tokenizer,
)

gpt_oss_length_bucket_morph_result["mean_summary_df"]


[✓] Tokenizer loaded from: gpt-oss/tokenizer


,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,permutation/gpt-oss_length_bucket_rand,output_proj,6,True,5,5,euclidean,eom,0.0,0.117157,0.058780,1203
1,permutation/gpt-oss_length_bucket_rand,output_proj,182,True,5,5,euclidean,eom,0.0,0.076798,0.037849,740
2,permutation/gpt-oss_length_bucket_rand,output_proj,466,True,5,5,euclidean,eom,0.0,0.105496,0.052407,1941
3,permutation/gpt-oss_length_bucket_rand,output_proj,739,True,5,5,euclidean,eom,0.0,0.110296,0.051800,2783
4,permutation/gpt-oss_length_bucket_rand,output_proj,1591,True,5,5,euclidean,eom,0.0,0.112005,0.048008,3625
5,permutation/gpt-oss_length_bucket_rand,output_proj,2264,True,5,5,euclidean,eom,0.0,0.111604,0.047979,3819
6,permutation/gpt-oss_length_bucket_rand,output_proj,2532,True,5,5,euclidean,eom,0.0,0.111332,0.047547,3856
7,permutation/gpt-oss_length_bucket_rand,output_proj,2868,True,5,5,euclidean,eom,0.0,0.111951,0.047345,3880
8,permutation/gpt-oss_length_bucket_rand,output_proj,2880,True,5,5,euclidean,eom,0.0,0.112073,0.047429,3880


# Random Orthogonal Projection Morphology Analysis

In [5]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

from random_projection import DEFAULT_SEEDS, MODEL_CONFIGS
from kondrak_cluster_morphology import (
    aggregate_permutation_morphology_means,
    concat_global_summary_rows,
    run_kondrak_morphology_for_partition,
    save_permutation_morphology_summary_df,
)


def run_random_projection_morphology_for_model(
    model_key,
    *,
    out_root="comp",
    space_name="output_proj",
    seed_list=DEFAULT_SEEDS,
):
    model_config = MODEL_CONFIGS[model_key]
    model_name = model_config["model_name"]
    pca_dim_list = model_config["candidate_dims"]
    tokenizer = AutoTokenizer.from_pretrained(model_config["tokenizer_path"])
    random_projection_dir = Path(out_root) / model_name / space_name / "random_projection"
    per_seed_summary = {}
    all_seed_rows = []

    for rp_seed in seed_list:
        global_rows = []
        summary_filename = f"random_projection/seed_{int(rp_seed)}/summary.csv"
        seed_dir = random_projection_dir / f"seed_{int(rp_seed)}"

        for pca_dim in pca_dim_list:
            cluster_outpath = (
                seed_dir / "morph" / f"kondrak_clusters_pca_{int(pca_dim)}.csv"
            )
            _, global_row = run_kondrak_morphology_for_partition(
                out_root=out_root,
                model_name=model_name,
                space_name=space_name,
                pca_dim=int(pca_dim),
                l2_norm=True,
                min_cluster_size=5,
                min_samples=5,
                metric="euclidean",
                cluster_selection_method="eom",
                cluster_selection_epsilon=0.0,
                summary_filename=summary_filename,
                print_columns=False,
                save_cluster_csv=True,
                tokenizer=tokenizer,
                cluster_csv_outpath=cluster_outpath,
            )
            global_row = global_row.copy()
            global_row["rp_seed"] = int(rp_seed)
            global_rows.append(global_row)

        seed_summary_df = concat_global_summary_rows(global_rows)
        seed_summary_df = seed_summary_df.sort_values("pca_dim").reset_index(drop=True)
        seed_summary_path = seed_dir / "kondrak_global_summary.csv"
        save_permutation_morphology_summary_df(seed_summary_df, seed_summary_path)
        per_seed_summary[int(rp_seed)] = seed_summary_df
        all_seed_rows.append(seed_summary_df)

    all_seed_summary_df = pd.concat(all_seed_rows, axis=0, ignore_index=True)
    all_seed_summary_df = all_seed_summary_df.sort_values(
        ["rp_seed", "pca_dim"]
    ).reset_index(drop=True)
    mean_summary_df = aggregate_permutation_morphology_means(all_seed_summary_df)

    save_permutation_morphology_summary_df(
        all_seed_summary_df,
        random_projection_dir / "kondrak_global_summary_all_seeds.csv",
    )
    save_permutation_morphology_summary_df(
        mean_summary_df,
        random_projection_dir / "kondrak_global_summary_mean.csv",
    )
    return {
        "per_seed_summary": per_seed_summary,
        "all_seed_summary_df": all_seed_summary_df,
        "mean_summary_df": mean_summary_df,
    }

In [6]:
random_projection_morphology_results = {}
for model_key in ["mistral", "mixtral", "gpt-oss"]:
    random_projection_morphology_results[model_key] = (
        run_random_projection_morphology_for_model(model_key)
    )

random_projection_morphology_results["gpt-oss"]["mean_summary_df"]

,model_name,space_name,pca_dim,l2_norm,min_cluster_size,min_samples,metric,cluster_selection_method,cluster_selection_epsilon,global_mean,global_std,n_clusters
0,gpt-oss,output_proj,6,True,5,5,euclidean,eom,0.0,0.102733,0.050020,3259
1,gpt-oss,output_proj,182,True,5,5,euclidean,eom,0.0,0.535485,0.177210,3345
2,gpt-oss,output_proj,466,True,5,5,euclidean,eom,0.0,0.534868,0.179094,3680
3,gpt-oss,output_proj,739,True,5,5,euclidean,eom,0.0,0.534717,0.178765,3768
4,gpt-oss,output_proj,1591,True,5,5,euclidean,eom,0.0,0.534395,0.179704,3859
5,gpt-oss,output_proj,2264,True,5,5,euclidean,eom,0.0,0.534232,0.179163,3864
6,gpt-oss,output_proj,2532,True,5,5,euclidean,eom,0.0,0.534220,0.179431,3863
7,gpt-oss,output_proj,2868,True,5,5,euclidean,eom,0.0,0.534275,0.179429,3875
8,gpt-oss,output_proj,2880,True,5,5,euclidean,eom,0.0,0.533921,0.179170,3877
